## ARIMA Model

ARIMA = AR + I + MA

Where:
- AR = past values
- I = differencing (to make data stationary)
- MA = past errors



### Why ARIMA?
Real-world data is usually non-stationary

So we:
* Step 1: Make data stationary
* Step 2: Apply AR + MA



### Example:
Sales data with trend → use ARIMA

<img src="ARIMA_working.png" alt="Time Series info" width="900" height="600" />


In [9]:
from IPython.display import HTML
HTML("""
<style>
.CodeMirror pre {
    white-space: pre-wrap;
}
</style>
""")

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error
import warnings

warnings.filterwarnings("ignore")

In [2]:
# Fetch Real Dataset from the Internet
print("--- Fetching Real Machine Sensor Data ---")
url = "https://raw.githubusercontent.com/numenta/NAB/master/data/realKnownCause/machine_temperature_system_failure.csv"

# Reading CSV data from the internet
df = pd.read_csv(url)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.set_index('timestamp', inplace=True)

# Data is recorded every 5 minutes. To make ARIMA faster, we convert it into hourly average values.
df_hourly = df.resample('H').mean().dropna()
print(f"Total Hourly Data Points: {len(df_hourly)}\n")


--- Fetching Real Machine Sensor Data ---
Total Hourly Data Points: 1891



In [3]:
# Stationarity Check

print("--- Checking Stationarity ---")
result = adfuller(df_hourly['value'])
print(f"ADF Statistic: {result[0]:.4f}")
print(f"p-value: {result[1]:.4f}")
if result[1] < 0.05:
    print("Data is mostly stationary, but since this is real machine data, we can still use d=1 to be safe.\n")

--- Checking Stationarity ---
ADF Statistic: -6.2418
p-value: 0.0000
Data is mostly stationary, but since this is real machine data, we can still use d=1 to be safe.



In [4]:
# Train-Test Split
# First 200 hours are used for training, next 50 hours (around 2 days) for testing
train_data = df_hourly.iloc[:200]
test_data = df_hourly.iloc[200:250]

print(f"Training Data: {len(train_data)} hours")
print(f"Testing Data: {len(test_data)} hours\n")

Training Data: 200 hours
Testing Data: 50 hours



In [ ]:
# ARIMA Robust Rolling Forecast
print("--- Starting Rolling Forecast (Might take a minute) ---")

''' Before predicting the future, the model needs to know the past. 
Here, we are taking all the temperature values from training data and
putting them into a list called history '''
history = [x for x in train_data['value']]

# This is an empty bucket where we will store the model's future temperature predictions one by one.
predictions = []

for t in range(len(test_data)):
    try:
        # Check variance to avoid mathematical errors in real-world data
        if np.var(history) == 0:
            yhat = history[-1] 
        else:
            # ARIMA(p=2, d=1, q=2) is used to handle possible trends in machine temperature
            model = ARIMA(history, 
                          order=(2, 1, 2), 
                          enforce_stationarity=False, 
                          enforce_invertibility=False)
            
            # The innovations_mle method is just a specific, faster mathematical way to fit the model.
            fitted_model = model.fit(method='innovations_mle')
            yhat = fitted_model.forecast()[0]
            
    except Exception as e:
        # Fallback in case of unexpected errors in real-world data
        yhat = history[-1]

    predictions.append(yhat)
    
    # Update history with the actual observed value
    obs = test_data['value'].iloc[t]
    
    
    '''This is the secret of the rolling forecast. Instead of predicting 
    Hour 2 using only the training data, we add the actual temperature from
    Hour 1 into your history. When the loop restarts to predict Hour 2, 
    the model is smarter because it knows exactly what just happened in Hour 1.'''
    history.append(obs)
    
    if t % 10 == 0:
        print(f"Hour {t}: Predicted Temp={yhat:.2f}, Actual Temp={obs:.2f}")

predictions_series = pd.Series(predictions, index=test_data.index)


--- Starting Rolling Forecast (Might take a minute) ---
Hour 0: Predicted Temp=98.37, Actual Temp=101.12
Hour 10: Predicted Temp=93.22, Actual Temp=94.96
Hour 20: Predicted Temp=94.03, Actual Temp=94.63
Hour 30: Predicted Temp=92.15, Actual Temp=92.13
Hour 40: Predicted Temp=93.73, Actual Temp=93.69


In [16]:
test_data['value']

timestamp
2013-12-11 05:00:00    101.119584
2013-12-11 06:00:00    101.558640
2013-12-11 07:00:00     98.212440
2013-12-11 08:00:00     94.902490
2013-12-11 09:00:00     91.084940
2013-12-11 10:00:00     89.235451
2013-12-11 11:00:00     89.422640
2013-12-11 12:00:00     89.093725
2013-12-11 13:00:00     89.279440
2013-12-11 14:00:00     92.027982
2013-12-11 15:00:00     94.955052
2013-12-11 16:00:00     95.060270
2013-12-11 17:00:00     95.304355
2013-12-11 18:00:00     94.979250
2013-12-11 19:00:00     95.240982
2013-12-11 20:00:00     95.013200
2013-12-11 21:00:00     95.120643
2013-12-11 22:00:00     95.250465
2013-12-11 23:00:00     94.833287
2013-12-12 00:00:00     94.688913
2013-12-12 01:00:00     94.627502
2013-12-12 02:00:00     94.038388
2013-12-12 03:00:00     92.867047
2013-12-12 04:00:00     89.873543
2013-12-12 05:00:00     90.586221
2013-12-12 06:00:00     91.815371
2013-12-12 07:00:00     90.419909
2013-12-12 08:00:00     87.507094
2013-12-12 09:00:00     88.153080
2013

In [15]:
predictions_series

timestamp
2013-12-11 05:00:00     98.368997
2013-12-11 06:00:00    102.265898
2013-12-11 07:00:00    100.778195
2013-12-11 08:00:00     95.000296
2013-12-11 09:00:00     92.600551
2013-12-11 10:00:00     88.301097
2013-12-11 11:00:00     88.190220
2013-12-11 12:00:00     89.189595
2013-12-11 13:00:00     88.204262
2013-12-11 14:00:00     89.010661
2013-12-11 15:00:00     93.215611
2013-12-11 16:00:00     95.544813
2013-12-11 17:00:00     93.834858
2013-12-11 18:00:00     94.887476
2013-12-11 19:00:00     93.867572
2013-12-11 20:00:00     94.825607
2013-12-11 21:00:00     93.998730
2013-12-11 22:00:00     94.590084
2013-12-11 23:00:00     94.554633
2013-12-12 00:00:00     93.841100
2013-12-12 01:00:00     94.030407
2013-12-12 02:00:00     93.914405
2013-12-12 03:00:00     93.007373
2013-12-12 04:00:00     91.611980
2013-12-12 05:00:00     87.558205
2013-12-12 06:00:00     91.325686
2013-12-12 07:00:00     91.674535
2013-12-12 08:00:00     88.820051
2013-12-12 09:00:00     85.554412
2013

In [7]:
# Evaluation 
mse = mean_squared_error(test_data['value'], predictions)
print(f"\nMean Squared Error (MSE): {mse:.2f}")


Mean Squared Error (MSE): 1.70


We achieved a Mean Squared Error (MSE) of 1.70. 

Real-World Meaning: An RMSE of 1.30 means that, on average, our model's predictions are only off by about 1.3 degrees from the actual machine temperature.

Relative to the Data: The machine's temperature fluctuates widely between 50 and over 100 degrees. An error of just 1.3 degrees across such a large range is incredibly small.


In [8]:
# Plotly Interactive Dashboard
fig = go.Figure()

# Training data (historical values)
fig.add_trace(go.Scatter(x=train_data.index, y=train_data['value'], 
                         mode='lines', name='Historical Temp (Train)', line=dict(color='blue')))

# Actual test data
fig.add_trace(go.Scatter(x=test_data.index, y=test_data['value'], 
                         mode='lines+markers', name='Actual Temp (Test)', line=dict(color='orange')))

# Predicted values from ARIMA
fig.add_trace(go.Scatter(x=test_data.index, y=predictions_series, 
                         mode='lines', name='ARIMA Expected Temp', line=dict(color='red', width=2, dash='dash')))

fig.update_layout(title='Real IoT Data: Industrial Machine Temperature Forecast ',
                  xaxis_title='Date & Time', 
                  yaxis_title='Temperature Sensor Value',
                  template='plotly_white')
fig.show()



### This interactive graph visually validates the performance of our ARIMA rolling forecast against the real machine data. Here is how to read the visual elements:

- X-Axis (Date & Time): Displays the continuous timeline of the sensor readings, covering the training period (Dec 3 - Dec 11) and the testing period (Dec 11 - Dec 13).

- Y-Axis (Temperature Sensor Value): Represents the actual heat levels recorded by the industrial machine's IoT sensor.

- Solid Blue Line (Historical Temp - Train): This represents the past data. It shows the complex, non-stationary temperature patterns that the ARIMA model used to learn the machine's behavior.

- Solid Orange Line (Actual Temp - Test): This represents the true, real-world temperature readings that occurred during our testing window.

- Dashed Red Line (ARIMA Expected Temp): This shows the step-by-step predictions generated by our model.


` Key Takeaway `: The graph demonstrates a highly successful forecast. The dashed red line tightly overlaps the orange line, meaning the model's predictions closely mirrored reality. It successfully anticipated sharp drops and sudden spikes in the temperature, proving that the model adapted well to the real-world variance.

#### **What we did in this project:**

• Collected real IoT machine temperature data
• Resampled high-frequency data into hourly averages
• Performed stationarity check using ADF test
• Built an ARIMA(2,1,2) model for trend + noise handling
• Implemented a rolling forecast (real-world scenario simulation)
• Evaluated performance using Mean Squared Error (MSE)
• Visualized predictions vs actual values using an interactive dashboard

## ARIMA Use Cases in Manufacturing

1. Tool Wear Prediction

Cutting tools slowly wear out over time. ARIMA forecasts vibration or friction data to predict when a tool should be replaced before product quality decreases.

2. Emissions Forecasting

Factories must control pollution levels. ARIMA predicts future CO₂ or gas emissions using past data, helping industries stay within legal limits.

3. Thermal Drift in CNC Machines

Machines heat up during operation, affecting precision. ARIMA forecasts temperature changes so systems can automatically adjust machine accuracy.

4. Coolant and Fluid Degradation

Industrial coolants lose effectiveness over time. ARIMA predicts changes in pH or viscosity, allowing timely fluid replacement before machine damage occurs.

<div style="text-align: center; font-size: 16px;">
<b>Created by</b><br>
Priyanka Deore & Kunal Mahadule
</div>